In [19]:
# ==========================================================
# Imports
# ==========================================================

import torch
import torch.nn as nn

import numpy as np
import pandas as pd

import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

In [20]:
# ==========================================================
# Load Training Dataset
# ==========================================================

TRAIN_CSV = "/Users/admin/Desktop/reliable_rejection_under_degradation/ablation_study/train_mlp_extended_final_plus_tinyface.csv"

df = pd.read_csv(TRAIN_CSV)

print("Dataset Shape :", df.shape)

display(df.head())

print("\nLabel Counts")
print(df["label"].value_counts())

print("\nLabel Distribution")
print(df["label"].value_counts(normalize=True))

Dataset Shape : (5117, 6)


,quality_score,best_similarity,margin,label,true_identity,gallery_identity
0,14.075935,0.248183,0.000138,0,NaN,NaN
1,15.297855,0.220784,0.031821,1,Jason_Momoa,Jason_Momoa
2,16.786978,-0.033517,-0.218560,1,Emma_Watson,Emma_Watson
3,13.985061,0.568350,0.314188,1,NaN,NaN
4,45.461479,0.201896,0.008333,0,NaN,NaN



Label Counts
label
1    2582
0    2535
Name: count, dtype: int64

Label Distribution
label
1    0.504593
0    0.495407
Name: proportion, dtype: float64


In [21]:
# ==========================================================
# Ablation Experiments
# ==========================================================

ABLATION_EXPERIMENTS = {
    "A1_similarity": ["best_similarity"],
    "A2_similarity_margin": ["best_similarity", "margin"],
    "A3_similarity_quality": ["best_similarity", "quality_score"],
    "A4_quality_margin": ["quality_score", "margin"],
    "A5_full_model": ["quality_score", "best_similarity", "margin"],
}

print("Total Experiments :", len(ABLATION_EXPERIMENTS))

for name, features in ABLATION_EXPERIMENTS.items():

    print(name, "->", features)

Total Experiments : 5
A1_similarity -> ['best_similarity']
A2_similarity_margin -> ['best_similarity', 'margin']
A3_similarity_quality -> ['best_similarity', 'quality_score']
A4_quality_margin -> ['quality_score', 'margin']
A5_full_model -> ['quality_score', 'best_similarity', 'margin']


In [22]:
# ==========================================================
# Multi Layer Perceptron
# ==========================================================


class MLPExperiment(nn.Module):

    def __init__(self, input_size):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )

    def forward(self, x):

        return self.network(x)

In [23]:
# ==========================================================
# Reproducibility
# ==========================================================

torch.manual_seed(42)

np.random.seed(42)

print("Random Seed Fixed")

Random Seed Fixed


In [24]:
MODEL_DIR = "saved_models"
SCALER_DIR = "saved_scalers"
import os
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(SCALER_DIR, exist_ok=True)

In [28]:
# ==========================================================
# Training Function
# ==========================================================


def train_ablation(feature_columns, experiment_name):

    print("\n" + "=" * 80)
    print(f"Training Experiment : {experiment_name}")
    print(f"Features            : {feature_columns}")
    print("=" * 80)

    # ------------------------------------------------------
    # Features and Labels
    # ------------------------------------------------------

    X = df[feature_columns]

    y = df["label"]

    print("\nLabel Counts")
    print(y.value_counts())

    # ------------------------------------------------------
    # Train Validation Split
    # ------------------------------------------------------

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y,
    )

    print("\nTrain Shape :", X_train.shape)
    print("Validation  :", X_val.shape)

    # ------------------------------------------------------
    # Scaling
    # ------------------------------------------------------

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)

    X_val_scaled = scaler.transform(X_val)


    joblib.dump(
        scaler,
        os.path.join(
            SCALER_DIR,
            f"{experiment_name}_scaler.pkl"
        )
        
    )

    print("Scaler Saved")

    # ------------------------------------------------------
    # Tensor Conversion
    # ------------------------------------------------------

    X_train_tensor = torch.tensor(
        X_train_scaled,
        dtype=torch.float32,
    )

    X_val_tensor = torch.tensor(
        X_val_scaled,
        dtype=torch.float32,
    )

    y_train_tensor = torch.tensor(
        y_train.values,
        dtype=torch.long,
    )

    y_val_tensor = torch.tensor(
        y_val.values,
        dtype=torch.long,
    )

    # ------------------------------------------------------
    # Model
    # ------------------------------------------------------

    model = MLPExperiment(input_size=len(feature_columns))

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.0005,
        weight_decay=1e-4,
    )

    print(model)

    # ------------------------------------------------------
    # Training
    # ------------------------------------------------------

    epochs = 300

    best_epoch = 0

    best_val_accuracy = -1.0

    for epoch in range(epochs):

        # ----------------------------
        # Train
        # ----------------------------

        model.train()

        outputs = model(X_train_tensor)

        loss = criterion(
            outputs,
            y_train_tensor,
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        # ----------------------------
        # Validation
        # ----------------------------

        model.eval()

        with torch.no_grad():

            val_outputs = model(X_val_tensor)

            predictions = torch.argmax(
                val_outputs,
                dim=1,
            )

            val_accuracy = accuracy_score(
                y_val_tensor.numpy(),
                predictions.numpy(),
            )

        # ----------------------------
        # Save Best Model
        # ----------------------------

        if val_accuracy > best_val_accuracy:

            best_val_accuracy = val_accuracy

            best_epoch = epoch

            torch.save(
                model.state_dict(),
                os.path.join(
                    MODEL_DIR,
                    f"{experiment_name}_model.pth"
                )
            )

        if epoch % 10 == 0:

            print(
                f"Epoch {epoch:3d}"
                f" | Loss {loss.item():.4f}"
                f" | Val Acc {val_accuracy:.4f}"
            )

    print()

    print("Best Validation Accuracy :", best_val_accuracy)

    print("Best Epoch :", best_epoch)
    
    # ------------------------------------------------------
    # Load Best Model
    # ------------------------------------------------------

    best_model = MLPExperiment(
        input_size=len(feature_columns)
    )

    best_model.load_state_dict(
        torch.load(
            os.path.join(
                MODEL_DIR,
                f"{experiment_name}_model.pth",
            ),
            map_location=torch.device("cpu"),
        )
    )

    best_model.eval()

    print("Best Model Loaded")

    # ------------------------------------------------------
    # Validation Prediction
    # ------------------------------------------------------

    with torch.no_grad():
        outputs = best_model(X_val_tensor)

        predictions = torch.argmax(
            outputs,
            dim=1,
        )

    y_pred = predictions.numpy()
    y_true = y_val_tensor.numpy()

    # ------------------------------------------------------
    # Metrics
    # ------------------------------------------------------

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print("\nValidation Results")
    print("------------------------------")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")

    # ------------------------------------------------------
    # Confusion Matrix
    # ------------------------------------------------------

    cm = confusion_matrix(y_true, y_pred)

    print("\nConfusion Matrix")
    print(cm)

    print("\nClassification Report")
    print(classification_report(
        y_true,
        y_pred,
        zero_division=0,
    ))

    TN, FP, FN, TP = cm.ravel()

    TAR = TP / (TP + FN)
    FRR = FN / (TP + FN)
    TRR = TN / (TN + FP)
    FAR = FP / (TN + FP)

    print("\nAdditional Metrics")
    print(f"TAR : {TAR:.4f}")
    print(f"FRR : {FRR:.4f}")
    print(f"TRR : {TRR:.4f}")
    print(f"FAR : {FAR:.4f}")

    # ------------------------------------------------------
    # Return Results
    # ------------------------------------------------------

    return {
        "Experiment": experiment_name,
        "Features": ", ".join(feature_columns),
        "Num Features": len(feature_columns),
        "Best Epoch": best_epoch,
        "Val Accuracy": accuracy,
        "Val Precision": precision,
        "Val Recall": recall,
        "Val F1": f1,
        "Val TAR": TAR,
        "Val FRR": FRR,
        "Val TRR": TRR,
        "Val FAR": FAR,
        "TN": TN,
        "FP": FP,
        "FN": FN,
        "TP": TP,
    }

In [29]:
# ==========================================================
# Train All Ablation Models
# ==========================================================

all_results = []

print("=" * 80)
print("Starting Ablation Study")
print("=" * 80)

for experiment_name, feature_columns in ABLATION_EXPERIMENTS.items():

    result = train_ablation(
        feature_columns=feature_columns,
        experiment_name=experiment_name,
    )

    all_results.append(result)

print("\nTraining Complete!")

training_results = pd.DataFrame(all_results)

training_results = training_results[
    [
        "Experiment",
        "Features",
        "Num Features",
        "Best Epoch",
        "Val Accuracy",
        "Val Precision",
        "Val Recall",
        "Val F1",
        "Val TAR",
        "Val FRR",
        "Val TRR",
        "Val FAR",
    ]
]

training_results.to_csv(
    "ablation_training_summary.csv",
    index=False,
)

print("\nTraining Summary")

display(training_results)

Starting Ablation Study

Training Experiment : A1_similarity
Features            : ['best_similarity']

Label Counts
label
1    2582
0    2535
Name: count, dtype: int64

Train Shape : (4093, 1)
Validation  : (1024, 1)
Scaler Saved
MLPExperiment(
  (network): Sequential(
    (0): Linear(in_features=1, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=8, bias=True)
    (5): ReLU()
    (6): Linear(in_features=8, out_features=2, bias=True)
  )
)
Epoch   0 | Loss 0.6853 | Val Acc 0.5049
Epoch  10 | Loss 0.6680 | Val Acc 0.5049
Epoch  20 | Loss 0.6508 | Val Acc 0.5049
Epoch  30 | Loss 0.6339 | Val Acc 0.5049
Epoch  40 | Loss 0.6170 | Val Acc 0.5400
Epoch  50 | Loss 0.6008 | Val Acc 0.8740
Epoch  60 | Loss 0.5840 | Val Acc 0.8916
Epoch  70 | Loss 0.5689 | Val Acc 0.8984
Epoch  80 | Loss 0.5527 | Val Acc 0.9004
Epoch  90 | Loss 0.5356 | Val Acc 0.9014
Epoch 100 | Loss 0.5175 | Val

,Experiment,Features,Num Features,Best Epoch,Val Accuracy,Val Precision,Val Recall,Val F1,Val TAR,Val FRR,Val TRR,Val FAR
0,A1_similarity,best_similarity,1,299,0.927734,0.974304,0.880077,0.924797,0.880077,0.119923,0.976331,0.023669
1,A2_similarity_margin,"best_similarity, margin",2,250,0.966797,0.982036,0.951644,0.966601,0.951644,0.048356,0.982249,0.017751
2,A3_similarity_quality,"best_similarity, quality_score",2,290,0.916992,0.988688,0.845261,0.911366,0.845261,0.154739,0.990138,0.009862
3,A4_quality_margin,"quality_score, margin",2,263,0.916016,0.979955,0.851064,0.910973,0.851064,0.148936,0.982249,0.017751
4,A5_full_model,"quality_score, best_similarity, margin",3,233,0.925781,0.986755,0.864603,0.921649,0.864603,0.135397,0.988166,0.011834


In [30]:
# checking saved model
import os

print("Saved Models\n")

for exp in ABLATION_EXPERIMENTS:

    model_file = os.path.join(
        MODEL_DIR,
        f"{exp}_model.pth"
    )

    scaler_file = os.path.join(
        SCALER_DIR,
        f"{exp}_scaler.pkl"
    )

    print(model_file, os.path.exists(model_file))
    print(scaler_file, os.path.exists(scaler_file))
    print("-"*50)

Saved Models

saved_models/A1_similarity_model.pth True
saved_scalers/A1_similarity_scaler.pkl True
--------------------------------------------------
saved_models/A2_similarity_margin_model.pth True
saved_scalers/A2_similarity_margin_scaler.pkl True
--------------------------------------------------
saved_models/A3_similarity_quality_model.pth True
saved_scalers/A3_similarity_quality_scaler.pkl True
--------------------------------------------------
saved_models/A4_quality_margin_model.pth True
saved_scalers/A4_quality_margin_scaler.pkl True
--------------------------------------------------
saved_models/A5_full_model_model.pth True
saved_scalers/A5_full_model_scaler.pkl True
--------------------------------------------------
